# End-to-End Pipeline Demo

Runs the full agent-based model sequence on ~10 agents:
synthesize → persona → work/school zone → activities → destinations →
schedule → tours → mode choice.

**Requires** an `ANTHROPIC_API_KEY` environment variable.

## 1. Imports + client setup

In [11]:
from collections import Counter

from aibm import ZoneSpec, synthesize_population
from aibm.agent import ModeOption
from aibm.llm import AnthropicClient
from aibm.zone import Zone

client = AnthropicClient()
print("Client ready.")

Client ready.


## 2. Define zones

Three zones for our toy model:
- **north** — residential suburb
- **centre** — commercial downtown
- **east** — mixed: residential, commercial, and school

In [2]:
zones = [
    Zone(
        id="north",
        name="North Suburb",
        x=0.0,
        y=1.0,
        land_use={"residential": True, "commercial": False, "school": False},
    ),
    Zone(
        id="centre",
        name="City Centre",
        x=0.0,
        y=0.0,
        land_use={"residential": False, "commercial": True, "school": False},
    ),
    Zone(
        id="east",
        name="East Side",
        x=1.0,
        y=0.0,
        land_use={"residential": True, "commercial": True, "school": True},
    ),
]

zone_lookup = {z.id: z for z in zones}
print(f"{len(zones)} zones defined.")

3 zones defined.


## 3. Define travel times

Dict mapping `(origin, destination)` to `{mode: minutes}`.

In [3]:
travel_times = {
    ("north", "north"): {"car": 5, "transit": 10, "bike": 8, "walk": 15},
    ("north", "centre"): {"car": 15, "transit": 25, "bike": 30, "walk": 60},
    ("north", "east"): {"car": 20, "transit": 35, "bike": 40, "walk": 80},
    ("centre", "north"): {"car": 15, "transit": 25, "bike": 30, "walk": 60},
    ("centre", "centre"): {"car": 5, "transit": 8, "bike": 7, "walk": 12},
    ("centre", "east"): {"car": 10, "transit": 20, "bike": 25, "walk": 50},
    ("east", "north"): {"car": 20, "transit": 35, "bike": 40, "walk": 80},
    ("east", "centre"): {"car": 10, "transit": 20, "bike": 25, "walk": 50},
    ("east", "east"): {"car": 5, "transit": 10, "bike": 8, "walk": 15},
}


def get_travel_times_from(home: str) -> dict[str, dict[str, float]]:
    """Get travel times from a home zone to all other zones."""
    return {dest: modes for (orig, dest), modes in travel_times.items() if orig == home}


print("Travel times defined for all zone pairs.")

Travel times defined for all zone pairs.


## 4. Synthesize ~10 agents

Two small zones, a few households each.

In [4]:
zone_specs = [
    ZoneSpec(
        zone_id="north",
        n_households=3,
        household_size_dist={1: 0.2, 2: 0.5, 3: 0.3},
        age_dist={"0-17": 0.15, "18-64": 0.70, "65+": 0.15},
        employment_rate=0.70,
        student_rate=0.10,
        vehicle_dist={0: 0.1, 1: 0.6, 2: 0.3},
        license_rate=0.85,
    ),
    ZoneSpec(
        zone_id="east",
        n_households=3,
        household_size_dist={1: 0.3, 2: 0.4, 3: 0.3},
        age_dist={"0-17": 0.10, "18-64": 0.75, "65+": 0.15},
        employment_rate=0.50,
        student_rate=0.25,
        vehicle_dist={0: 0.4, 1: 0.45, 2: 0.15},
        license_rate=0.65,
    ),
]

households = synthesize_population(zone_specs, seed=42)
all_agents = [agent for hh in households for agent in hh.members]

for agent in all_agents:
    agent.model = "claude-haiku-4-5-20251001"

print(f"Households: {len(households)}, Agents: {len(all_agents)}")
for hh in households:
    print(f"  {hh}  zone={hh.home_zone}  vehicles={hh.num_vehicles}")
    for a in hh.members:
        lic = "licence" if a.has_license else "no licence"
        print(f"    {a.name}  age={a.age}  {a.employment}  {lic}")

Households: 6, Agents: 12
  Household(id='ededf33d-74c1-453f-9278-875580d9d022', size=2)  zone=north  vehicles=0
    Agent 0  age=24  employed  no licence
    Agent 1  age=13  unemployed  no licence
  Household(id='62d496e7-db72-434c-9832-905b81946c8b', size=1)  zone=north  vehicles=0
    Agent 2  age=53  employed  licence
  Household(id='d790241a-8a71-4ea6-b40c-0c159cf16c42', size=2)  zone=north  vehicles=1
    Agent 3  age=18  student  licence
    Agent 4  age=35  employed  no licence
  Household(id='6f720875-21cc-4c4e-a304-5f6e60daf412', size=2)  zone=east  vehicles=0
    Agent 5  age=56  employed  licence
    Agent 6  age=25  unemployed  licence
  Household(id='7f757730-fba3-49b3-9923-5b5cb73e6ad9', size=2)  zone=east  vehicles=1
    Agent 7  age=83  retired  licence
    Agent 8  age=7  unemployed  no licence
  Household(id='5c034bc6-61f9-4d3a-9fac-51783418c600', size=3)  zone=east  vehicles=2
    Agent 9  age=77  retired  licence
    Agent 10  age=41  employed  licence
    Agent 1

## 5. Generate personas

In [12]:
hh_lookup = {a.id: hh for hh in households for a in hh.members}

for agent in all_agents:
    hh = hh_lookup[agent.id]
    agent.generate_persona(client=client, household=hh)
    print(f"{agent.name}: {agent.persona}")

Agent 0: Agent 0 is a budget-conscious 24-year-old who navigates their north zone commute and daily errands entirely through public transit, walking, and cycling, relying on affordable buses and trains to reach their workplace while minimizing transportation costs. Their travel patterns are concentrated within the northern neighborhood, with trips strategically planned around transit schedules and bike-sharing availability to maintain mobility despite lacking a driving license and household vehicle.
Agent 1: A 13-year-old student from the north zone who commutes to school and local activities entirely via public transit, walking, and cycling, with travel patterns confined to pedestrian-friendly routes and public transit corridors within their neighborhood due to lack of household vehicles and driving capability. Their daily routine is structured around school schedules and nearby community centers, with journeys typically short-distance and dependent on accessible public transportation

## 6. Choose work/school zones

In [14]:
for agent in all_agents:
    home = agent.home_zone
    assert home is not None
    tt = get_travel_times_from(home)

    if agent.employment == "employed":
        zone_id = agent.choose_work_zone(zones, tt, client=client)
        print(f"{agent.name} works in: {zone_id}")
    elif agent.employment == "student":
        zone_id = agent.choose_school_zone(zones, tt, client=client)
        print(f"{agent.name} studies in: {zone_id}")
    else:
        print(f"{agent.name}: no work/school zone needed ({agent.employment})")

Agent 0 works in: north
Agent 1: no work/school zone needed (unemployed)
Agent 2 works in: north
Agent 3 studies in: north
Agent 4 works in: north
Agent 5 works in: east
Agent 6: no work/school zone needed (unemployed)
Agent 7: no work/school zone needed (retired)
Agent 8: no work/school zone needed (unemployed)
Agent 9: no work/school zone needed (retired)
Agent 10 works in: centre
Agent 11: no work/school zone needed (unemployed)


## 7. Generate activities

In [15]:
agent_activities: dict[str, list] = {}

for agent in all_agents:
    activities = agent.generate_activities(client=client)
    agent_activities[agent.id] = activities
    types = [a.type for a in activities]
    print(f"{agent.name}: {types}")

Agent 0: ['work', 'grocery shopping', 'lunch', 'commute home', 'leisure/relaxation']
Agent 1: ['school', 'homework', 'leisure - cycling with friends', 'shopping - groceries or supplies', 'community center activity']
Agent 2: ['work', 'grocery shopping', 'meal preparation', 'leisure/relaxation']
Agent 3: ['School', 'Library study session', 'Grocery shopping', 'Coffee with friends', 'Lunch']
Agent 4: ['work', 'grocery shopping', 'lunch', 'household chores', 'leisure/relaxation']
Agent 5: ['work', 'grocery shopping', 'commute via public transportation', 'meal preparation', 'leisure/relaxation']
Agent 6: ['job search', 'social service appointment', 'grocery shopping', 'leisure - visit local park']
Agent 7: ['Medical appointment', 'Grocery shopping', 'Household errands', 'Social visit']
Agent 8: ['school', 'lunch at home', 'park play', 'snack time', 'family errands with parent', 'dinner with family']
Agent 9: ['breakfast', 'shopping', 'lunch', 'social visit', 'cultural event', 'dining']
Age

## 8. Choose destinations (flexible activities only)

In [16]:
for agent in all_agents:
    for activity in agent_activities[agent.id]:
        if activity.is_flexible and activity.location is None:
            agent.choose_destination(activity, candidates=zones, client=client)
            print(f"{agent.name} → {activity.type} at {activity.location}")

Agent 0 → grocery shopping at north
Agent 0 → lunch at north
Agent 0 → leisure/relaxation at north
Agent 1 → homework at north
Agent 1 → leisure - cycling with friends at north
Agent 1 → shopping - groceries or supplies at north
Agent 1 → community center activity at north
Agent 2 → grocery shopping at north
Agent 2 → meal preparation at north
Agent 2 → leisure/relaxation at north
Agent 3 → Library study session at north
Agent 3 → Grocery shopping at north
Agent 3 → Coffee with friends at north
Agent 3 → Lunch at north
Agent 4 → grocery shopping at north
Agent 4 → lunch at north
Agent 4 → household chores at north
Agent 4 → leisure/relaxation at centre
Agent 5 → grocery shopping at east
Agent 5 → meal preparation at east
Agent 5 → leisure/relaxation at east
Agent 6 → job search at centre
Agent 6 → grocery shopping at east
Agent 6 → leisure - visit local park at east
Agent 7 → Grocery shopping at east
Agent 7 → Household errands at east
Agent 7 → Social visit at east
Agent 8 → lunch at 

## 9. Schedule activities

In [17]:
agent_plans: dict[str, object] = {}

for agent in all_agents:
    activities = agent_activities[agent.id]
    day_plan = agent.schedule_activities(activities, client=client)
    agent_plans[agent.id] = day_plan
    print(f"\n{agent.name}'s schedule:")
    for act in day_plan.activities:
        print(
            f"  {act.type:12s}  "
            f"{act.start_time:6.0f}-{act.end_time:6.0f}  "
            f"@ {act.location}"
        )


Agent 0's schedule:
  lunch            720-   780  @ north
  work             780-  1020  @ north
  grocery shopping    1020-  1080  @ north
  commute home    1080-  1140  @ None
  leisure/relaxation    1140-  1380  @ north

Agent 1's schedule:
  school           480-   900  @ None
  homework         900-  1020  @ north
  shopping - groceries or supplies    1020-  1080  @ north
  leisure - cycling with friends    1080-  1200  @ north
  community center activity    1200-  1320  @ north

Agent 2's schedule:
  work             480-   900  @ north
  grocery shopping     900-   960  @ north
  meal preparation     960-  1020  @ north
  leisure/relaxation    1020-  1320  @ north

Agent 3's schedule:
  School           480-   720  @ None
  Lunch            720-   780  @ north
  Library study session     780-   960  @ north
  Coffee with friends     960-  1020  @ north
  Grocery shopping    1020-  1080  @ north

Agent 4's schedule:
  lunch            720-   780  @ north
  work             780-

## 10. Build tours

In [18]:
for agent in all_agents:
    day_plan = agent_plans[agent.id]
    agent.build_tours(day_plan)
    n_tours = len(day_plan.tours)
    n_trips = len(day_plan.trips)
    print(f"\n{agent.name}: {n_tours} tour(s), {n_trips} trip(s)")
    for i, tour in enumerate(day_plan.tours):
        print(f"  Tour {i + 1} (closed={tour.is_closed}):")
        for trip in tour.trips:
            print(f"    {trip.origin} → {trip.destination}")

ValueError: Activity 'commute home' has no location

## 11. Choose mode per trip

In [ ]:
all_mode_choices = []

for agent in all_agents:
    hh = hh_lookup[agent.id]
    day_plan = agent_plans[agent.id]

    for trip in day_plan.trips:
        pair = (trip.origin, trip.destination)
        tt = travel_times.get(pair, {"walk": 30})

        # Build mode options, filtering car if no licence or 0 vehicles
        options = []
        for mode, minutes in tt.items():
            if mode == "car" and (not agent.has_license or hh.num_vehicles == 0):
                continue
            options.append(ModeOption(mode=mode, travel_time=minutes))

        if not options:
            options = [ModeOption(mode="walk", travel_time=30)]

        choice = agent.choose_mode(options, client=client, household=hh)
        trip.mode = choice.option.mode
        all_mode_choices.append(choice)
        print(
            f"{agent.name}: {trip.origin}→{trip.destination} "
            f"by {trip.mode} ({choice.option.travel_time:.0f} min)"
        )

## 12. Summary — mode share

In [ ]:
all_trips = [trip for agent in all_agents for trip in agent_plans[agent.id].trips]

mode_counts = Counter(t.mode for t in all_trips)
total = len(all_trips)

print(f"Total trips: {total}\n")
print("Mode share:")
for mode, count in mode_counts.most_common():
    print(f"  {mode:8s}: {count:3d}  ({count / total:.0%})")